# Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7 & 9) ingin membangun dashboard pemantauan pesanan real-time untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Anda ditugaskan membangun purwarupa (prototype) pipeline streaming yang mensimulasikan kebutuhan ini.

In [1]:
# Membuat SparkSession
import spark_session

app = spark_session.SparkApp("Tugas10-PipelineStream")
spark = app.getSession()
spark.conf.set("spark.sql.shuffle.partitions", "4")

Memulai SparkSession untuk Tugas10-PipelineStream


26/11/09 09:08:17 WARN Utils: Your hostname, asfadani resolves to a loopback address: 127.0.1.1; using 192.168.18.43 instead (on interface wlx088af1bd9610)
26/11/09 09:08:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 09:08:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


### A. Menyiapkan Streaming Source (bobot 15%)

Definisikan skema eksplisit (StructType) untuk data pesanan dengan kolom: pesanan_id (string), nama_cabang (string), nama_menu (string), qty (integer), harga_satuan (integer), waktu_pesan (timestamp). Buat folder simulasi streaming baru bernama tugas10_stream, lalu definisikan readStream dari folder tersebut.

In [7]:
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, avg, col
import pandas as pd
import shutil, os, time

In [3]:
folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

skema = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

print("siap untuk tugas", folder_tugas)

siap untuk tugas tugas10_stream


In [4]:
# Mendefinisikan readstream
df_stream_tugas = spark.readStream \
    .format("csv") \
    .schema(skema) \
    .option("header", True) \
    .load("file:///home/asfadani/praktikum-bigdata/Praktikum-BigData/tugas10_stream")

print("Streaming DataFrame berhasil didefinisikan (belum berjalan - baru 'rencana').")
print("df_stream_latihan.isStreaming:", df_stream_tugas.isStreaming)

Streaming DataFrame berhasil didefinisikan (belum berjalan - baru 'rencana').
df_stream_latihan.isStreaming: True


## B. Agregasi per Cabang (bobot 20%)

Tambahkan kolom turunan total_harga (qty x harga_satuan) pada df_stream, lalu buat agregasi groupBy("nama_cabang") yang menghitung total total_harga dan jumlah pesanan. Jalankan sebagai streaming query dengan sink memory, queryName="agregasi_cabang".

In [5]:
# Menambahkan kolom total harga
df_stream_tugas = df_stream_tugas.withColumn("total_harga", col("qty") * col("harga_satuan"))

agg_tugas = df_stream_tugas.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_tugas = agg_tugas.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming query dimulai! Status:", query_tugas.status)

Streaming query dimulai! Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}


## C. Agregasi Berbasis Window Waktu (bobot 20%)

Buat streaming query kedua (definisikan ulang readStream dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total total_harga berdasarkan window waktu 2 menit dari kolom waktu_pesan. Gunakan sink memory, queryName="agregasi_waktu".


In [13]:
# Mendefinisikan radStream
df_stream_waktu = spark.readStream\
    .format("csv")\
    .schema(skema)\
    .option("header", True)\
    .load("file:///home/asfadani/praktikum-bigdata/Praktikum-BigData/tugas10_stream")

df_stream_waktu = df_stream_waktu.withColumn("total_harga", col("qty") * col("harga_satuan"))

# membuat query ke 2
agg_window = df_stream_waktu.groupBy(
    "nama_cabang", window(col("waktu_pesan"), "2 minute")).agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_waktu = agg_window.writeStream\
    .outputMode("complete")\
    .format("memory")\
    .queryName("agregasi_waktu")\
    .start()

print("Streaming query dimulai! Status:", query_waktu.status)

IllegalArgumentException: Cannot start query with name agregasi_waktu as a query with that name is already active in this SparkSession

Kode diatas error karena saya sebelumnya membuat kueri aggregasi yang mengelompokkan berdasarkan window waktu. Namun kemudian setelah kueri saya nyalakan saya kepikiran untuk menambah mengelompokkan berdasarkan cabangnya juga. Nah karena saya menambahkan nama_cabang nya setelah kueri berjalan maka akan error karena seperti menyalakan lagi kueri yang sudah nyala namun berbeda isi kueri. Oleh karena itu saya ingin mematikan kuerinya yang sebelumnya baru menyalakan lagi yang ada nama cabangnya,

In [14]:
# Mencari kueri aktif bernama 'agregasi_waktu' lalu menghentikannya
for s in spark.streams.active:
    if s.name == "agregasi_waktu":
        s.stop()

In [6]:
# Mendefinisikan radStream
df_stream_waktu = spark.readStream\
    .format("csv")\
    .schema(skema)\
    .option("header", True)\
    .load("file:///home/asfadani/praktikum-bigdata/Praktikum-BigData/tugas10_stream")

df_stream_waktu = df_stream_waktu.withColumn("total_harga", col("qty") * col("harga_satuan"))

# membuat query ke 2
agg_window = df_stream_waktu.groupBy(
    "nama_cabang", window(col("waktu_pesan"), "2 minute")).agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_waktu = agg_window.writeStream\
    .outputMode("complete")\
    .format("memory")\
    .queryName("agregasi_waktu")\
    .start()

print("Streaming query dimulai! Status:", query_waktu.status)

Streaming query dimulai! Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}


Nah, dapat dilihaty bahwa sekarang sudh berhasil nyala.

## D. Menguji dengan Data Simulasi (bobot 25%)

Kirim minimal 3 batch data (masing-masing batch berupa satu berkas CSV baru dengan jeda time.sleep() di antaranya) ke folder tugas10_stream, dengan kombinasi nama_cabang yang bervariasi. Setelah setiap batch, tampilkan hasil agregasi_cabang untuk membuktikan angka totalnya terus terakumulasi dengan benar.

In [8]:
# mengirim data batch 1

# Waktu saat ini untuk simulasi data real-time
now = datetime.now()

pd.DataFrame({
    "pesanan_id": ["KN-001", "KN-002", "KN-003", "KN-004", "KN-005", "KN-006", "KN-007", "KN-008"],
    "nama_cabang": [
        "Kopi Nusantara Magelang", 
        "Kopi Nusantara Jogja", 
        "Kopi Nusantara Solo", 
        "Kopi Nusantara Semarang", 
        "Kopi Nusantara Magelang", 
        "Kopi Nusantara Jogja", 
        "Kopi Nusantara Semarang", 
        "Kopi Nusantara Solo"
    ],
    "nama_menu": [
        "Kopi Susu Gula Aren", 
        "Matcha Latte", 
        "Butterscotch Coffee", 
        "Americano", 
        "Manual Brew V60", 
        "Matcha Latte", 
        "Espresso", 
        "Butterscotch Coffee"
    ],
    "qty": [2, 1, 3, 1, 2, 1, 4, 2],
    "harga_satuan": [20000, 25000, 28000, 18000, 22000, 25000, 15000, 28000],
    "waktu_pesan": [
        (now - timedelta(minutes=45)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(minutes=32)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(minutes=25)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(minutes=20)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(minutes=15)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(minutes=10)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(minutes=5)).strftime("%Y-%m-%d %H:%M:%S"),
        now.strftime("%Y-%m-%d %H:%M:%S")
    ]
}).to_csv(f"{folder_tugas}/kopi_nusantara_batch_1.csv", index=False)

print("Batch 1 dikirim. Menunggu Spark memproses...")
time.sleep(6)

spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

Batch 1 dikirim. Menunggu Spark memproses...


+--------------------+----------------+--------------+
|         nama_cabang|total_pendapatan|jumlah_pesanan|
+--------------------+----------------+--------------+
| Kopi Nusantara Solo|          140000|             2|
|Kopi Nusantara Ma...|           84000|             2|
|Kopi Nusantara Se...|           78000|             2|
|Kopi Nusantara Jogja|           50000|             2|
+--------------------+----------------+--------------+



In [10]:
# mengirim batch 2
now = datetime.now()

pd.DataFrame({
    "pesanan_id": ["KN-009", "KN-010", "KN-011", "KN-012", "KN-013", "KN-014", "KN-015", "KN-016"],
    "nama_cabang": [
        "Kopi Nusantara Solo", 
        "Kopi Nusantara Semarang", 
        "Kopi Nusantara Magelang", 
        "Kopi Nusantara Jogja", 
        "Kopi Nusantara Jogja", 
        "Kopi Nusantara Solo", 
        "Kopi Nusantara Semarang", 
        "Kopi Nusantara Magelang"
    ],
    "nama_menu": [
        "Americano", 
        "Kopi Susu Gula Aren", 
        "Espresso", 
        "Butterscotch Coffee", 
        "Manual Brew V60", 
        "Matcha Latte", 
        "Kopi Susu Gula Aren", 
        "Americano"
    ],
    "qty": [2, 3, 1, 2, 1, 2, 4, 1],
    "harga_satuan": [18000, 20000, 15000, 28000, 22000, 25000, 20000, 18000],
    "waktu_pesan": [
        (now - timedelta(minutes=8)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(minutes=7)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(minutes=5)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(minutes=4)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(minutes=3)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(minutes=2)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S"),
        now.strftime("%Y-%m-%d %H:%M:%S")
    ]
}).to_csv(f"{folder_tugas}/kopi_nusantara_batch_2.csv", index=False)

print("Batch 2 dikirim. Menunggu Spark memproses...")
time.sleep(6)

spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

Batch 2 dikirim. Menunggu Spark memproses...
+--------------------+----------------+--------------+
|         nama_cabang|total_pendapatan|jumlah_pesanan|
+--------------------+----------------+--------------+
| Kopi Nusantara Solo|          226000|             4|
|Kopi Nusantara Se...|          218000|             4|
|Kopi Nusantara Jogja|          128000|             4|
|Kopi Nusantara Ma...|          117000|             4|
+--------------------+----------------+--------------+



In [11]:
spark.sql("SELECT * FROM agregasi_waktu ORDER BY total_pendapatan DESC").show()

+--------------------+--------------------+----------------+--------------+
|         nama_cabang|              window|total_pendapatan|jumlah_pesanan|
+--------------------+--------------------+----------------+--------------+
| Kopi Nusantara Solo|{2026-11-09 09:20...|           92000|             2|
| Kopi Nusantara Solo|{2026-11-09 08:56...|           84000|             1|
|Kopi Nusantara Se...|{2026-11-09 09:28...|           80000|             1|
|Kopi Nusantara Se...|{2026-11-09 09:16...|           60000|             1|
|Kopi Nusantara Se...|{2026-11-09 09:22...|           60000|             1|
|Kopi Nusantara Jogja|{2026-11-09 09:24...|           56000|             1|
| Kopi Nusantara Solo|{2026-11-09 09:26...|           50000|             1|
|Kopi Nusantara Ma...|{2026-11-09 09:06...|           44000|             1|
|Kopi Nusantara Ma...|{2026-11-09 08:36...|           40000|             1|
|Kopi Nusantara Jogja|{2026-11-09 09:10...|           25000|             1|
|Kopi Nusant

In [12]:
# Mengirim batch 3
now = datetime.now()

pd.DataFrame({
    "pesanan_id": ["KN-017", "KN-018", "KN-019", "KN-020", "KN-021", "KN-022", "KN-023", "KN-024"],
    "nama_cabang": [
        "Kopi Nusantara Semarang", 
        "Kopi Nusantara Jogja", 
        "Kopi Nusantara Jogja", 
        "Kopi Nusantara Magelang", 
        "Kopi Nusantara Solo", 
        "Kopi Nusantara Semarang", 
        "Kopi Nusantara Solo", 
        "Kopi Nusantara Magelang"
    ],
    "nama_menu": [
        "Espresso", 
        "Matcha Latte", 
        "Butterscotch Coffee", 
        "Kopi Susu Gula Aren", 
        "Americano", 
        "Manual Brew V60", 
        "Matcha Latte", 
        "Butterscotch Coffee"
    ],
    "qty": [1, 2, 1, 3, 2, 1, 1, 2],
    "harga_satuan": [15000, 25000, 28000, 20000, 18000, 22000, 25000, 28000],
    "waktu_pesan": [
        (now - timedelta(seconds=240)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(seconds=200)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(seconds=150)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(seconds=120)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(seconds=90)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(seconds=60)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(seconds=30)).strftime("%Y-%m-%d %H:%M:%S"),
        now.strftime("%Y-%m-%d %H:%M:%S")
    ]
}).to_csv(f"{folder_tugas}/kopi_nusantara_batch_3.csv", index=False)

print("Batch 3 dikirim. Menunggu Spark memproses...")
time.sleep(6)

spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

Batch 3 dikirim. Menunggu Spark memproses...
+--------------------+----------------+--------------+
|         nama_cabang|total_pendapatan|jumlah_pesanan|
+--------------------+----------------+--------------+
| Kopi Nusantara Solo|          287000|             6|
|Kopi Nusantara Se...|          255000|             6|
|Kopi Nusantara Ma...|          233000|             6|
|Kopi Nusantara Jogja|          206000|             6|
+--------------------+----------------+--------------+



In [13]:
spark.sql("SELECT * FROM agregasi_waktu ORDER BY total_pendapatan DESC").show()

+--------------------+--------------------+----------------+--------------+
|         nama_cabang|              window|total_pendapatan|jumlah_pesanan|
+--------------------+--------------------+----------------+--------------+
|Kopi Nusantara Se...|{2026-11-09 09:28...|           95000|             2|
| Kopi Nusantara Solo|{2026-11-09 09:20...|           92000|             2|
| Kopi Nusantara Solo|{2026-11-09 08:56...|           84000|             1|
|Kopi Nusantara Ma...|{2026-11-09 09:30...|           60000|             1|
|Kopi Nusantara Se...|{2026-11-09 09:22...|           60000|             1|
|Kopi Nusantara Se...|{2026-11-09 09:16...|           60000|             1|
|Kopi Nusantara Ma...|{2026-11-09 09:32...|           56000|             1|
|Kopi Nusantara Jogja|{2026-11-09 09:24...|           56000|             1|
| Kopi Nusantara Solo|{2026-11-09 09:26...|           50000|             1|
|Kopi Nusantara Jogja|{2026-11-09 09:28...|           50000|             1|
|Kopi Nusant

## E. Sink Berbasis Berkas (Tantangan) (bobot 20%)

Selain memory sink, Structured Streaming juga mendukung menulis hasil DataFrame mentah (bukan hasil agregasi) ke berkas secara berkelanjutan. Buat streaming query ketiga yang membaca dari df_stream, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal tugas10_output dalam format Parquet, menggunakan:

In [28]:
for q in spark.streams.active:
    q.stop()

In [31]:
path_stream = "file:///home/asfadani/praktikum-bigdata/Praktikum-BigData/tugas10_stream"
path_output = "file:///home/asfadani/praktikum-bigdata/Praktikum-BigData/tugas10_output"
path_checkpoint = "file:///home/asfadani/praktikum-bigdata/Praktikum-BigData/tugas10_checkpoint"

df_stream_parquet = spark.readStream \
    .format("csv") \
    .schema(skema) \
    .option("header", True) \
    .load(path_stream)

In [32]:
query3 = df_stream_parquet.writeStream \
    .format("parquet") \
    .option("path", path_output) \
    .option("checkpointLocation", path_checkpoint) \
    .outputMode("append") \
    .start()

print("Status Stream:", query3.status)

Status Stream: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (!ls tugas10_output) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan spark.read.parquet().

In [19]:
# batch tambahan
now = datetime.now()

pd.DataFrame({
    "pesanan_id": ["KN-025", "KN-026", "KN-027", "KN-028", "KN-029", "KN-030", "KN-031", "KN-032"],
    "nama_cabang": [
        "Kopi Nusantara Semarang", 
        "Kopi Nusantara Solo", 
        "Kopi Nusantara Magelang", 
        "Kopi Nusantara Jogja", 
        "Kopi Nusantara Magelang", 
        "Kopi Nusantara Semarang", 
        "Kopi Nusantara Jogja", 
        "Kopi Nusantara Solo"
    ],
    "nama_menu": [
        "Butterscotch Coffee", 
        "Espresso", 
        "Kopi Susu Gula Aren", 
        "Manual Brew V60", 
        "Matcha Latte", 
        "Americano", 
        "Kopi Susu Gula Aren", 
        "Butterscotch Coffee"
    ],
    "qty": [2, 1, 3, 1, 2, 1, 2, 1],
    "harga_satuan": [28000, 15000, 20000, 22000, 25000, 18000, 20000, 28000],
    "waktu_pesan": [
        (now - timedelta(seconds=115)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(seconds=95)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(seconds=80)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(seconds=60)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(seconds=45)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(seconds=30)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(seconds=15)).strftime("%Y-%m-%d %H:%M:%S"),
        now.strftime("%Y-%m-%d %H:%M:%S")
    ]
}).to_csv(f"{folder_tugas}/kopi_nusantara_batch_1.csv", index=False)

print("Batch tambahan dikirim dikirim. Menunggu Spark memproses...")

Batch tambahan dikirim dikirim. Menunggu Spark memproses...


In [33]:
now = datetime.now()

pd.DataFrame({
    "pesanan_id": ["KN-666", "KN-999"],
    "nama_cabang": ["Kopi Nusantara Magelang", "Kopi Nusantara Jogja"],
    "nama_menu": ["Kopi Susu Gula Aren", "Americano"],
    "qty": [1, 2],
    "harga_satuan": [20000, 18000],
    "waktu_pesan": [
        (now - timedelta(seconds=115)).strftime("%Y-%m-%d %H:%M:%S"),
        (now - timedelta(seconds=95)).strftime("%Y-%m-%d %H:%M:%S")
    ]
}).to_csv(f"{folder_tugas}/test_ls_2.csv", index=False)

print("Batch tambahan dikirim dikirim. Menunggu Spark memproses...")

Batch tambahan dikirim dikirim. Menunggu Spark memproses...


In [34]:
!ls /home/asfadani/praktikum-bigdata/Praktikum-BigData/tugas10_output

part-00000-5c6a61bd-1e97-4858-8c39-37220a361d40-c000.snappy.parquet
part-00000-9c167549-aa7f-40eb-9602-8e055e505a1f-c000.snappy.parquet
part-00001-32d8b964-3103-47dd-8881-702ac73b3e75-c000.snappy.parquet
part-00002-eb3ff30a-16ac-4063-84eb-a856cd562e17-c000.snappy.parquet
part-00003-337b2003-261b-4d74-b7bf-9eab10387c77-c000.snappy.parquet
_spark_metadata
